# Paired webcam + lidar viewer

Interactive viewer for acquisition outputs (`<run>/images/*.jpg` + `<run>/lidar/*.npy`).
Set `RUN_DIR`, then run the cells. Scrub with the slider at the bottom.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.image import imread
from ipywidgets import IntSlider, FloatSlider, Dropdown, interact

# Import project-local helpers
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "examples"))

from examples.plot_paired import pair_files, render_lidar  # noqa: E402

In [ ]:
# >>> edit this to point at an acquisition run <<<
RUN_DIR = Path("/home/rfor10/lidar_data/20260421_demo_raw_backpack")

pairs = pair_files(RUN_DIR)
print(f"{len(pairs)} paired frames in {RUN_DIR}")
assert pairs, "no paired files found — check RUN_DIR"

In [ ]:
def show(idx, view, gamma, clip_hi_pct):
    img_path, lid_path = pairs[idx]
    img = imread(img_path)
    pts = np.load(lid_path)
    lidar_img, cmap = render_lidar(pts, view, gamma=gamma, clip_hi_pct=clip_hi_pct)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(img)
    axes[0].set_title(f"webcam: {img_path.name}")
    axes[0].axis("off")

    axes[1].imshow(lidar_img, cmap=cmap)
    axes[1].set_title(f"lidar ({view}) γ={gamma:.2f} clip={clip_hi_pct:.0f}%  [{pts.shape[0]} pts]")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()


interact(
    show,
    idx=IntSlider(min=0, max=len(pairs) - 1, step=1, value=0, description="frame"),
    view=Dropdown(options=["front", "pano", "bev"], value="front", description="view"),
    gamma=FloatSlider(min=0.2, max=2.0, step=0.05, value=0.5, description="gamma"),
    clip_hi_pct=FloatSlider(min=80.0, max=100.0, step=0.5, value=99.0, description="clip hi %"),
);

## Save a montage of N sampled frames

In [ ]:
from examples.plot_paired import plot_pairs

stride = max(1, len(pairs) // 6)
plot_pairs(pairs[::stride][:6], view="front", save=None)